In [1]:
import os
import sys
import numpy as np
import gymnasium as gym
from sumo_rl import SumoEnvironment
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import pickle

I0000 00:00:1775774344.623301   56021 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775774347.638178   56021 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
NET_FILE = "./cross.net.xml"
ROU_FILE = "./cross.rou.xml"

if not NET_FILE or not ROU_FILE:
    print(f"CRITICAL ERROR: 'cross.net.xml' or 'cross.rou.xml' not found")
    sys.exit()


In [3]:
class QLearningAgent:
    """
    Q-Learning reinforcement learning agent.
    
    This agent uses a Q-table to learn optimal policies for traffic signal control.
    """
    def __init__(self, action_size):
        self.q_table = {}
        self.lr, self.gamma, self.epsilon = 0.1, 0.95, 1.0
        self.action_size = action_size

    def get_state_key(self, state):
        return tuple(np.round(state, 1))

    def choose_action(self, state):
        state_key = self.get_state_key(state)
        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.action_size)
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        return np.argmax(self.q_table[state_key])

    def learn(self, s, a, r, s_next):
        s_k, sn_k = self.get_state_key(s), self.get_state_key(s_next)
        if sn_k not in self.q_table: 
            self.q_table[sn_k] = np.zeros(self.action_size)
            
        predict = self.q_table[s_k][a]
        # Q-LEARNING UPDATE: Use the maximum possible Q-value for the next state
        target = r + self.gamma * np.max(self.q_table[sn_k])
        
        self.q_table[s_k][a] += self.lr * (target - predict)
    
    def decay_epsilon(self, decay_rate=0.995, min_epsilon=0.01):
        """Decay the exploration rate."""
        self.epsilon = max(min_epsilon, self.epsilon * decay_rate)

    def save_model(self, filepath="qlearning_qtable.pkl"):
        """Saves the Q-table to a file."""
        with open(filepath, 'wb') as f:
            pickle.dump(self.q_table, f)
        print(f"Model successfully saved to: {filepath}")

    def load_model(self, filepath="qlearning_qtable.pkl"):
        """Loads the Q-table from a file."""
        if os.path.exists(filepath):
            with open(filepath, 'rb') as f:
                self.q_table = pickle.load(f)
            print(f"Model successfully loaded from: {filepath}")
        else:
            print(f"Warning: No saved model found at {filepath}")

In [4]:
def evaluate_model(agent, net_file, rou_file, tripinfo_file="tripinfo.xml"):
    """
    Evaluates the trained agent and generates a tripinfo.xml file.
    """
    print(f"\n--- Starting Evaluation ---")
    print(f"Generating trip info file at: {tripinfo_file}")
    
    # Initialize a fresh environment specifically for evaluation
    eval_env = SumoEnvironment(
        net_file=net_file, 
        route_file=rou_file,
        use_gui=False, # Set to True if you want to watch the trained agent
        num_seconds=3600,
        single_agent=True,
        additional_sumo_cmd=f"--tripinfo-output {tripinfo_file}" # Tells SUMO to generate the XML
    )
    
    state, info = eval_env.reset()
    done = False
    total_eval_reward = 0
    
    # Save the original epsilon and set it to 0 for pure exploitation (no random actions)
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, info = eval_env.step(action)
        
        done = terminated or truncated
        state = next_state
        total_eval_reward += reward
        
    # Clean up and restore
    agent.epsilon = original_epsilon
    eval_env.close()
    
    print(f"Evaluation completed! Total Reward: {total_eval_reward}")
    print(f"Trip info successfully saved to: {os.path.abspath(tripinfo_file)}\n")


In [5]:
# Initialize environment with the 4-way intersection files
env = SumoEnvironment(
    net_file=NET_FILE, 
    route_file=ROU_FILE,
    use_gui=False,
    num_seconds=3600,
    single_agent=True,
    sumo_warnings = False
)

 Retrying in 1 seconds
Step #0.00 (0ms ?*RT. ?UPS, TraCI: 6ms, vehicles TOT 0 ACT 0 BUF 0)                      


In [6]:
# Environment will now detect the traffic light phases from cross.net.xml
action_size = env.action_space.n
agent = QLearningAgent(action_size=action_size)

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir='./qlearning_tensorboard')

print("Starting Q-Learning Training...")
episodes = 50
reward_history = []

for e in range(episodes):
    state, info = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        # 1. Choose action for current state
        action = agent.choose_action(state)
        
        # 2. Take action, observe environment
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # 3. Learn from the transition (no next_action needed!)
        agent.learn(state, action, reward, next_state)
        
        # 4. Move to the next state
        state = next_state
        total_reward += reward
    
    reward_history.append(total_reward)
    # Log to TensorBoard
    writer.add_scalar('Reward/Episode', total_reward, e + 1)
    writer.add_scalar('Epsilon', agent.epsilon, e + 1)
    # Decay epsilon
    agent.decay_epsilon()
    print(f"Episode {e + 1} | Total Shared Reward: {total_reward:.2f}")

writer.close()
print("Training completed successfully.")

Starting Q-Learning Training...
 Retrying in 1 seconds
Episode 1 | Total Shared Reward: -8.13
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 36ms, vehicles TOT 1311 ACT 48 BUF 0)               
 Retrying in 1 seconds
Episode 2 | Total Shared Reward: -5.92
Step #3600.00 (1ms ~= 1000.00*RT, ~50000.00UPS, TraCI: 22ms, vehicles TOT 1311 ACT 50 BUF 
 Retrying in 1 seconds
Episode 3 | Total Shared Reward: -20.58
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 51ms, vehicles TOT 1311 ACT 65 BUF 0)               
 Retrying in 1 seconds
Episode 4 | Total Shared Reward: -1.92
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 30ms, vehicles TOT 1311 ACT 44 BUF 0)               
 Retrying in 1 seconds
Episode 5 | Total Shared Reward: -2.00
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 20ms, vehicles TOT 1311 ACT 56 BUF 0)               
 Retrying in 1 seconds
Episode 6 | Total Shared Reward: -28.19
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 64ms, vehicles TOT 1311 ACT 75 BUF 0)               
 Retrying in 1 seconds
Episode 7 | Total Shared 

In [7]:
env.close()

Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 503959ms, vehicles TOT 1311 ACT 43 BUF 0)           


In [8]:
agent.save_model()

Model successfully saved to: qlearning_qtable.pkl


In [9]:
evaluate_model(agent, NET_FILE, ROU_FILE, tripinfo_file="tripinfo_eval.xml")


--- Starting Evaluation ---
Generating trip info file at: tripinfo_eval.xml
 Retrying in 1 seconds
Step #0.00 (0ms ?*RT. ?UPS, TraCI: 4ms, vehicles TOT 0 ACT 0 BUF 0)                      
 Retrying in 1 seconds
Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 15ms, vehicles TOT 1311 ACT 44 BUF 0)               
Evaluation completed! Total Reward: -2.220446049250313e-16
Trip info successfully saved to: /home/ash/projects/Intelligent-Traffic-Signal-Control-using-Reinforcement-Learning/ql/tripinfo_eval.xml

